Task: Create a MLflow pipeline based on your final project, then zip the artifact of the experiment and submit in Binusmaya.

This notebook demonstrates the MLflow tracking pipeline using the **same algorithms, feature engineering (`src/features.py`), data cleaning, and encoding** as the final deployed project. Two deliberate simplifications were made purely to keep MLflow demo runs fast and reproducible for submission:

- **Model A** trains on a **100k-row representative subset** of the 1M-row dataset (vs. the full dataset in `notebooks/modelA.ipynb`), with the *same* hyperparameters, feature engineering, IQR cleaning, and SMOTE balancing as the compact deployed model.
- **Model B** uses a **fixed hyperparameter configuration** (logged via `mlflow.log_params`) instead of re-running the `RandomizedSearchCV` search performed in `notebooks/modelB.ipynb`.

In [5]:
import mlflow
import os
import sys
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold,
    StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier, XGBRegressor

REPO_ROOT = os.path.abspath(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.features import engineer_features_a

DATA_PATH_A = os.path.join(REPO_ROOT, "datasets", "academic_stress_level.csv")
DATA_PATH_B = os.path.join(REPO_ROOT, "datasets", "student_lifestyle_dataset.csv")

mlflow.set_experiment("Academic Shield - Burnout & GPA Prediction")

<Experiment: artifact_location='file:///c:/Users/asus/Downloads/AcademicShield/mlruns/1', creation_time=1782563304095, experiment_id='1', last_update_time=1782563304095, lifecycle_stage='active', name='Academic Shield - Burnout & GPA Prediction', tags={}, trace_location=None, workspace='default'>

In [6]:
with mlflow.start_run(run_name="Model_A_XGBoost_Classifier"):

    print("Model A - XGBoost Classifier")

    mlflow.set_tags({
        "model_type":     "XGBoost Classifier",
        "task":           "Burnout Classification",
        "dataset":        "academic_stress_level.csv (1M rows)",
        "target":         "burnout_class {0 = Healthy, 1 = Mildly Burnout, 2 = Burnout}",
        "thresholds":     "(score < 4) = Healthy, (4-7) = Mild, (>= 7) = Burnout",
        "features":       "10 base features → 20 features after feature engineering",
        "cleaning":       "IQR on feature columns only (preserve minority class)",
        "balancing":      "SMOTE on training set only",
        "primary_metric": "Macro F1",
        "tuning":         "Manual (same fixed config as notebooks/modelA.ipynb - compact retrain)",
        "sampling":       "100k-row representative subset of the 1M-row dataset, sampled with random_state=42 - chosen for MLflow demo run speed; notebooks/modelA.ipynb (source of truth for models/modelA.pkl) trains on the full dataset with identical hyperparameters/feature-engineering/cleaning/balancing",
        "team":           "Group 3 - Andrew/Raynald/Adrian",
    })

    df_a = pd.read_csv(DATA_PATH_A)
    df_a = df_a.sample(n=100000, random_state=42).reset_index(drop=True)
    cols_a = [
        "study_hours_per_day", "sleep_hours", "exam_pressure", "stress_level",
        "financial_stress", "social_support", "anxiety_score", "depression_score",
        "family_expectation", "physical_activity", "burnout_score",
    ]
    df_a = df_a[cols_a]

    def bin_burnout(score):
        if score < 4:
            return 0
        elif score < 7:
            return 1
        else:
            return 2

    df_a["burnout_class"] = df_a["burnout_score"].apply(bin_burnout)
    df_a = df_a.drop(columns=["burnout_score"])

    print(f"Class distribution:\n{df_a['burnout_class'].value_counts().to_string()}")

    feature_cols_a = [c for c in df_a.columns if c != "burnout_class"]
    for col in feature_cols_a:
        q1 = df_a[col].quantile(0.25)
        q3 = df_a[col].quantile(0.75)
        iqr = q3 - q1
        df_a = df_a[(df_a[col] >= q1 - 1.5 * iqr) & (df_a[col] <= q3 + 1.5 * iqr)]
    print(f"After cleaning: {len(df_a)} rows")
    print(f"Class distribution after cleaning:\n{df_a['burnout_class'].value_counts().to_string()}")

    X_a = engineer_features_a(df_a.drop(columns=["burnout_class"]))
    y_a = df_a["burnout_class"]
    print(f"Features: {X_a.shape[1]}")

    X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
        X_a, y_a, test_size=0.2, random_state=42, stratify=y_a
    )
    smote = SMOTE(random_state=42)
    X_train_a_sm, y_train_a_sm = smote.fit_resample(X_train_a, y_train_a)
    print(f"Train before SMOTE: {X_train_a.shape[0]} rows")
    print(f"Train after SMOTE : {X_train_a_sm.shape[0]} rows")
    print(f"Test              : {X_test_a.shape[0]} rows")

    best_params_a = {
        "n_estimators":  200,
        "max_depth":     5,
        "learning_rate": 0.15,
        "objective":     "multi:softprob",
        "num_class":     3,
        "tree_method":   "hist",
        "random_state":  42,
    }
    mlflow.log_params({
        **best_params_a,
        "train_test_split":  "80/20 stratified",
        "smote":             "applied to training set only",
        "iqr_cleaning":      "features only",
        "n_features_base":   10,
        "n_features_total":  20,
        "original_size":     1000000,
        "sampled_size":      100000,
    })

    model_a = XGBClassifier(**best_params_a, n_jobs=-1, verbosity=0)
    model_a.fit(X_train_a_sm, y_train_a_sm)

    y_pred_a = model_a.predict(X_test_a)

    test_acc_a  = accuracy_score(y_test_a, y_pred_a)
    test_f1_a   = f1_score(y_test_a, y_pred_a, average="macro")
    test_prec_a = precision_score(y_test_a, y_pred_a, average="macro", zero_division=0)
    test_rec_a  = recall_score(y_test_a, y_pred_a, average="macro", zero_division=0)

    report_a = classification_report(
        y_test_a,
        y_pred_a,
        target_names=["Healthy", "Mildly Burnout", "Burnout"],
        output_dict=True,
    )

    skf      = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_f1_a  = cross_val_score(model_a, X_a, y_a, cv=skf, scoring="f1_macro")
    cv_acc_a = cross_val_score(model_a, X_a, y_a, cv=skf, scoring="accuracy")

    dummy_acc_a = y_a.value_counts(normalize=True).max()
    dummy_f1_a  = 1 / 3

    mlflow.log_metrics({
        "test_accuracy":        round(test_acc_a, 4),
        "test_macro_f1":        round(test_f1_a, 4),
        "test_precision_macro": round(test_prec_a, 4),
        "test_recall_macro":    round(test_rec_a, 4),
        "f1_healthy":           round(report_a["Healthy"]["f1-score"], 4),
        "f1_mild":              round(report_a["Mildly Burnout"]["f1-score"], 4),
        "f1_burnout":           round(report_a["Burnout"]["f1-score"], 4),
        "cv_accuracy_mean":     round(cv_acc_a.mean(), 4),
        "cv_accuracy_std":      round(cv_acc_a.std(), 4),
        "cv_macro_f1_mean":     round(cv_f1_a.mean(), 4),
        "cv_macro_f1_std":      round(cv_f1_a.std(), 4),
        "dummy_accuracy":       round(float(dummy_acc_a), 4),
        "dummy_macro_f1":       round(dummy_f1_a, 4),
    })

    mlflow.xgboost.log_model(model_a, name="model_a")

    print(f"\n  Test Accuracy  : {test_acc_a:.4f}")
    print(f"  Test Macro F1  : {test_f1_a:.4f}")
    print(f"  CV Macro F1    : {cv_f1_a.mean():.4f} ± {cv_f1_a.std():.4f}")
    print(f"  Dummy baseline : {dummy_acc_a:.4f} acc / ~{dummy_f1_a:.2f} F1")
    print("\nModel A logged successfully.")

Model A - XGBoost Classifier
Class distribution:
burnout_class
0    88630
1    10997
2      373
After cleaning: 98126 rows
Class distribution after cleaning:
burnout_class
0    87675
1    10247
2      204
Features: 20
Train before SMOTE: 78500 rows
Train after SMOTE : 210417 rows
Test              : 19626 rows

  Test Accuracy  : 0.8599
  Test Macro F1  : 0.5652
  CV Macro F1    : 0.5397 ± 0.0059
  Dummy baseline : 0.8935 acc / ~0.33 F1

Model A logged successfully.


In [7]:
FEATURE_ORDER_B = ["study_hours", "eca_hours", "sleep_hours", "social_hours", "physical_hours", "stress_level"]

with mlflow.start_run(run_name="Model_B_XGBRegressor"):

    print("Model B - XGBRegressor")

    mlflow.set_tags({
        "model_type":     "XGBRegressor",
        "task":           "GPA Regression",
        "dataset":        "student_lifestyle_dataset.csv (~2000 rows)",
        "target":         "GPA (0.0 - 4.0)",
        "cleaning":       "Z-score threshold = 3 (conservative for small dataset)",
        "encoding":       "stress_level: Low = 0, Moderate = 1, High = 2",
        "primary_metric": "R2",
        "tuning":         "Fixed config logged here for traceability — derived from the RandomizedSearchCV (20 iter, 3-fold, scoring=r2) run in notebooks/modelB.ipynb, the source of truth for models/modelB.pkl. Not re-searched in this run to keep the MLflow demo fast/reproducible.",
        "note":           "Algorithm changed from GradientBoostingRegressor — m2cgen does not support sklearn GBR",
        "team":           "Group 3 - Andrew/Raynald/Adrian",
    })

    df_b = pd.read_csv(DATA_PATH_B)
    df_b = df_b.drop(columns=["Student_ID"])
    df_b = df_b.rename(columns={
        "Study_Hours_Per_Day":             "study_hours",
        "Extracurricular_Hours_Per_Day":   "eca_hours",
        "Sleep_Hours_Per_Day":             "sleep_hours",
        "Social_Hours_Per_Day":            "social_hours",
        "Physical_Activity_Hours_Per_Day": "physical_hours",
        "Stress_Level":                    "stress_level",
        "GPA":                             "gpa",
    })

    print(f"Rows: {len(df_b)}")

    df_b = df_b[(np.abs(stats.zscore(df_b.select_dtypes("number"))) < 3).all(axis=1)]
    print(f"After cleaning: {len(df_b)} rows")

    df_b["stress_level"] = df_b["stress_level"].map({"Low": 0, "Moderate": 1, "High": 2})

    X_b = df_b[FEATURE_ORDER_B]
    y_b = df_b["gpa"]
    X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
        X_b, y_b, test_size=0.3, random_state=42
    )

    params_b = {
        "n_estimators":     400,
        "max_depth":        3,
        "learning_rate":    0.01,
        "subsample":        0.7,
        "colsample_bytree": 0.8,
        "reg_lambda":       2.0,
        "min_child_weight": 5,
        "random_state":     42,
    }

    mlflow.log_params({
        **params_b,
        "train_test_split":    "70/30",
        "cleaning_method":     "Z-score threshold = 3",
        "encoding":            "manual mapping (Low = 0/Moderate = 1/High = 2)",
        "n_features":          6,
        "dataset_size":        len(df_b),
        "feature_engineering": "None (simple model)",
        "tuning_source":       "RandomizedSearchCV in notebooks/modelB.ipynb (not re-run here)",
    })

    model_b = XGBRegressor(**params_b, n_jobs=-1, verbosity=0)
    model_b.fit(X_train_b, y_train_b)
    y_pred_b = model_b.predict(X_test_b)

    mse_b          = mean_squared_error(y_test_b, y_pred_b)
    rmse_b         = np.sqrt(mse_b)
    mae_b          = mean_absolute_error(y_test_b, y_pred_b)
    r2_b           = r2_score(y_test_b, y_pred_b)
    actual_range_b = 1.76  # full EDA range (min GPA 2.24, max 4.0)
    nrmse_b        = rmse_b / actual_range_b

    kf        = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2_b   = cross_val_score(model_b, X_b, y_b, cv=kf, scoring="r2")
    cv_mae_b  = cross_val_score(model_b, X_b, y_b, cv=kf, scoring="neg_mean_absolute_error")
    cv_rmse_b = cross_val_score(model_b, X_b, y_b, cv=kf, scoring="neg_root_mean_squared_error")

    mlflow.log_metrics({
        "test_mse":         round(mse_b, 4),
        "test_rmse":        round(rmse_b, 4),
        "test_mae":         round(mae_b, 4),
        "test_r2":          round(r2_b, 4),
        "nrmse_pct":        round(nrmse_b * 100, 2),
        "actual_gpa_range": round(float(actual_range_b), 4),
        "cv_r2_mean":       round(cv_r2_b.mean(), 4),
        "cv_r2_std":        round(cv_r2_b.std(), 4),
        "cv_mae_mean":      round((-cv_mae_b).mean(), 4),
        "cv_mae_std":       round((-cv_mae_b).std(), 4),
        "cv_rmse_mean":     round((-cv_rmse_b).mean(), 4),
        "cv_rmse_std":      round((-cv_rmse_b).std(), 4),
    })

    mlflow.xgboost.log_model(model_b, name="model_b")

    print(f"\n  Test R²    : {r2_b:.4f}")
    print(f"  Test RMSE  : {rmse_b:.4f}")
    print(f"  NRMSE      : {nrmse_b*100:.2f}%")
    print(f"  CV R²      : {cv_r2_b.mean():.4f} ± {cv_r2_b.std():.4f}")
    print("\nModel B logged successfully.")

Model B - XGBRegressor
Rows: 2000
After cleaning: 1996 rows

  Test R²    : 0.5357
  Test RMSE  : 0.2024
  NRMSE      : 11.50%
  CV R²      : 0.5283 ± 0.0248

Model B logged successfully.


In [8]:
print("  Academic Shield - MLflow Pipeline Summary")
print("  Experiment : Academic Shield - Burnout Classification & GPA Prediction")
print(f"  Run 1 (A)  : XGBoost Classifier")
print(f"               Test Macro F1 : {test_f1_a:.4f}")
print(f"               CV Macro F1   : {cv_f1_a.mean():.4f} +/- {cv_f1_a.std():.4f}")
print(f"  Run 2 (B)  : XGBRegressor")
print(f"               Test R²       : {r2_b:.4f}")
print(f"               NRMSE         : {nrmse_b*100:.2f}%")


  Academic Shield - MLflow Pipeline Summary
  Experiment : Academic Shield - Burnout Classification & GPA Prediction
  Run 1 (A)  : XGBoost Classifier
               Test Macro F1 : 0.5652
               CV Macro F1   : 0.5397 +/- 0.0059
  Run 2 (B)  : XGBRegressor
               Test R²       : 0.5357
               NRMSE         : 11.50%
